In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import confusion_matrix

# Load dataset
print("Loading dataset...")
df = pd.read_csv("construction_dataset.csv")
print("Shape:", df.shape)
print(df.head())
print("Missing values:\n", df.isnull().sum())

# Encode stage name and risk label
stage_encoder = LabelEncoder()
df["stage_encoded"] = stage_encoder.fit_transform(df["stage_name"])

label_encoder = LabelEncoder()
df["risk_label_encoded"] = label_encoder.fit_transform(df["risk_label"])

# ---------------- Classification ----------------
X_class = df[
    [
        "project_size_sqft",
        "total_project_budget",
        "stage_encoded",
        "total_stage_cost",
        "expected_stage_cost",
    ]
]
y_class = df["risk_label_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Classification accuracy:", acc)
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Feature importance
feat_imp = clf.feature_importances_
plt.figure(figsize=(10,5))
plt.barh(X_class.columns, feat_imp)
plt.title("Feature Importance")
plt.show()

# Cross validation
cv = cross_val_score(clf, X_class, y_class, cv=5)
print("Cross validation scores:", cv)
print("Average CV score:", cv.mean())


# ---------------- Regression ----------------
X_reg = df[
    [
        "project_size_sqft",
        "total_project_budget",
        "stage_encoded",
    ]
]
y_reg = df["expected_stage_cost"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg = RandomForestRegressor(random_state=42)
reg.fit(X_train_r, y_train_r)

y_pred_r = reg.predict(X_test_r)

mae = mean_absolute_error(y_test_r, y_pred_r)
mse = mean_squared_error(y_test_r, y_pred_r)
r2 = r2_score(y_test_r, y_pred_r)

print("Regression MAE:", mae)
print("Regression MSE:", mse)
print("Regression R2:", r2)

# ---------------- Save models ----------------
joblib.dump(clf, "risk_classifier.pkl")
joblib.dump(reg, "cost_regressor.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(stage_encoder, "stage_encoder.pkl")

print("Models saved successfully!")
